In [8]:

import pandas as pd
import numpy as np
from pathlib import Path

print("Loading dataset for Linear Regression (2nd Model)...")

df = pd.read_csv("../output/training_data_v6_no0min_trial.csv", encoding="utf-8-sig")
df = df.sort_values(["Player UUID", "season", "Gameweek"]).reset_index(drop=True)

print(f"   → Loaded {len(df):,} rows")
print("✔ Dataset sorted.")


Loading dataset for Linear Regression (2nd Model)...
   → Loaded 25,735 rows
✔ Dataset sorted.


In [9]:
print("Creating next-GW target...")

df["Target_NextGW"] = df.groupby(["Player UUID","season"])["Total Points"].shift(-1)

before = len(df)
df = df.dropna(subset=["Target_NextGW"]).reset_index(drop=True)
after = len(df)

print(f"🧹 Removed {before - after} rows (final GWs)")
print("✔ Target ready.")


Creating next-GW target...
🧹 Removed 2314 rows (final GWs)
✔ Target ready.


In [10]:

print(" Building last-5-observations validation split per player...")

df["Split"] = "Train"

for uuid, g in df.groupby("Player UUID", sort=False):
    df.loc[g.tail(5).index, "Split"] = "Validation"

print(df["Split"].value_counts())
print("✔ Split applied.")


 Building last-5-observations validation split per player...
Split
Train         19942
Validation     3479
Name: count, dtype: int64
✔ Split applied.


In [11]:

print("🔧 Selecting 8 features for explainable Linear Regression...")

selected_features = [
    "Avg_Total Points_L3",
    "Avg_Total Points_L5",
    "ICT Index",
    "Threat",
    "Influence",
    "Creativity",
    "Is Home",
    "Opponent Difficulty"
]

df_model = df[selected_features + ["Target_NextGW","Split"]].copy()

# Fix Is Home to numeric 0/1
df_model["Is Home"] = df_model["Is Home"].replace(
    {True:1, False:0, "True":1, "False":0}
).astype(int)

X = df_model[selected_features]
y = df_model["Target_NextGW"]

print("📦 Feature matrix shape:", X.shape)
print("✔ Feature selection complete.")


🔧 Selecting 8 features for explainable Linear Regression...
📦 Feature matrix shape: (23421, 8)
✔ Feature selection complete.


C:\Users\SOFI\AppData\Local\Temp\ipykernel_19488\2044576081.py:17: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_model["Is Home"] = df_model["Is Home"].replace(


In [12]:
# %%
print("Creating train/validation sets...")

train_mask = df_model["Split"] == "Train"
val_mask   = df_model["Split"] == "Validation"

X_train = X[train_mask].reset_index(drop=True)
X_val   = X[val_mask].reset_index(drop=True)

y_train = y[train_mask].reset_index(drop=True)
y_val   = y[val_mask].reset_index(drop=True)

print(f"→ Train rows: {len(X_train)}")
print(f"→ Val rows:   {len(X_val)}")
print("✔ Data split complete.")


Creating train/validation sets...
→ Train rows: 19942
→ Val rows:   3479
✔ Data split complete.


In [13]:

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("🤖 Training explainable Linear Regression model...")

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
y_train_pred = model.predict(X_train)
y_val_pred   = model.predict(X_val)

# Metrics
train_mae  = mean_absolute_error(y_train, y_train_pred)
val_mae    = mean_absolute_error(y_val, y_val_pred)

# RMSE (manual sqrt(MSE) because older sklearn doesn't support squared=False)
train_rmse = mean_squared_error(y_train, y_train_pred) ** 0.5
val_rmse   = mean_squared_error(y_val, y_val_pred) ** 0.5

# Output results
print("\n PERFORMANCE (Explainable Model):")
print(f"   • MAE Train:      {train_mae:.4f}")
print(f"   • MAE Validation: {val_mae:.4f}")
print(f"   • RMSE Train:     {train_rmse:.4f}")
print(f"   • RMSE Validation:{val_rmse:.4f}")

print("✔ Model trained.")



🤖 Training explainable Linear Regression model...

 PERFORMANCE (Explainable Model):
   • MAE Train:      2.1368
   • MAE Validation: 1.8602
   • RMSE Train:     2.9491
   • RMSE Validation:2.5416
✔ Model trained.


In [14]:

print(" Exporting model coefficients with signs...")

OUT_DIR = Path("../output/linear_regression_model_2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

coeff_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": model.coef_.round(6)
})

# sort by coefficient (negative → positive)
coeff_df = coeff_df.sort_values("Coefficient", ascending=True)

# add intercept
intercept_row = pd.DataFrame([{
    "Feature": "Intercept",
    "Coefficient": round(model.intercept_, 6)
}])

coeff_df = pd.concat([intercept_row, coeff_df], ignore_index=True)

coeff_df.to_csv(OUT_DIR / "coefficients.csv", index=False, encoding="utf-8-sig")

print("✔ Coefficients saved to linear_regression_model_2/coefficients.csv")
print(coeff_df)


 Exporting model coefficients with signs...
✔ Coefficients saved to linear_regression_model_2/coefficients.csv
               Feature  Coefficient
0            Intercept     1.717103
1              Is Home    -0.116658
2            Influence     0.001403
3           Creativity     0.005379
4  Avg_Total Points_L3     0.009198
5               Threat     0.011570
6            ICT Index     0.059479
7  Opponent Difficulty     0.070358
8  Avg_Total Points_L5     0.239554


In [15]:

print(" Saving predictions...")

df_out = df.copy()

df_out["Prediction_For_GW"] = df_out["Gameweek"] + 1
df_out["Total_Points_Actual"] = df_out["Target_NextGW"]
df_out["Total_Points_Predicted"] = model.predict(X).round(4)
df_out["Error"] = (df_out["Total_Points_Actual"] - df_out["Total_Points_Predicted"]).abs().round(4)

keep_cols = [
    "Player UUID", "Player Name", "Web Name",
    "season", "Prediction_For_GW",
    "Opponent Difficulty",
    "Total_Points_Actual", "Total_Points_Predicted",
    "Error", "Split"
]

df_final = df_out[keep_cols].copy()
df_final.rename(columns={"Split": "Train_or_Validation"}, inplace=True)

df_final.to_csv(OUT_DIR / "linear_reg_predictions.csv", index=False, encoding="utf-8-sig")

print("✔ Predictions saved.")


 Saving predictions...
✔ Predictions saved.


In [16]:

# PHASE — Update Model_Performance.csv (REPLACE existing entries)

print(" Updating Model_Performance.csv ...")

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_squared_error

MODEL_PERF_PATH = Path("../output/model_performance/Model_Performance.csv")


# Compute RMSE

train_rmse = mean_squared_error(y_train, y_train_pred) ** 0.5
val_rmse   = mean_squared_error(y_val, y_val_pred) ** 0.5


#Load existing table + baseline MAE

if MODEL_PERF_PATH.exists():
    perf = pd.read_csv(MODEL_PERF_PATH)

    # remove ONLY this model’s previous entries
    perf = perf[perf["Model"] != "Linear Regression (8 Features)"]

    # extract baseline validation MAE
    baseline_row = perf[perf["Model"] == "Rolling Average (TotalPoints)"]
    if len(baseline_row) > 0:
        baseline_val_mae = baseline_row.iloc[0]["MAE_Validation"]
    else:
        baseline_val_mae = val_mae
else:
    perf = pd.DataFrame()
    baseline_val_mae = val_mae


# Compute relative improvement

relative_improvement = (baseline_val_mae - val_mae) / baseline_val_mae
relative_improvement = round(relative_improvement, 5)


# NEW ROW FOR THIS MODEL

new_row = pd.DataFrame([{
    "Model": "Linear Regression (8 Features)",
    "MAE_Train": round(train_mae, 5),
    "MAE_Validation": round(val_mae, 5),
    "Relative_Improvement_vs_Baseline": relative_improvement,
    "RMSE_Train": round(train_rmse, 5),
    "RMSE_Validation": round(val_rmse, 5)
}])


# Append + Save clean

perf = pd.concat([perf, new_row], ignore_index=True)
perf.to_csv(MODEL_PERF_PATH, index=False, encoding="utf-8-sig")

print("✔ Model_Performance cleaned & updated successfully.")


 Updating Model_Performance.csv ...
✔ Model_Performance cleaned & updated successfully.


In [17]:
# ===================================================================
# REPORT — Players Removed Per Season (with Reason)
# ===================================================================

import pandas as pd
from pathlib import Path 

print("Generating report: Players Removed per Season (with reason)...\n")

# Load datasets at different stages
raw_df = pd.read_csv("../output/training_data_v6_positions_fixed.csv", encoding="utf-8-sig")
no0_df = pd.read_csv("../output/training_data_v6_no0min.csv", encoding="utf-8-sig")
trial_df = pd.read_csv("../output/training_data_v6_no0min_trial.csv", encoding="utf-8-sig")

# Keep only essential columns
raw_players = raw_df[["Player UUID", "Player Name", "season"]].drop_duplicates()
no0_players = no0_df[["Player UUID", "Player Name", "season"]].drop_duplicates()
trial_players = trial_df[["Player UUID", "Player Name", "season"]].drop_duplicates()

# ===========================================================
# STEP 1 — Detect players completely removed due to 0 minutes
# ===========================================================
removed_zero_minutes = raw_players.merge(
    no0_players,
    on=["Player UUID", "season"],
    how="left",
    suffixes=("_raw", "_no0")
)

removed_zero_minutes = removed_zero_minutes[removed_zero_minutes["Player Name_no0"].isna()]
removed_zero_minutes["Reason"] = "Minutes Played = 0"

# ===============================================================
# STEP 2 — Detect players removed AFTER removing last GWs (Target)
# ===============================================================
removed_after_target = no0_players.merge(
    trial_players,
    on=["Player UUID", "season"],
    how="left",
    suffixes=("_no0", "_trial")
)

removed_after_target = removed_after_target[removed_after_target["Player Name_trial"].isna()]
removed_after_target["Reason"] = "Removed due to no Target_NextGW"

# =============================================================
# STEP 3 — Detect players from excluded seasons (trial filtering)
# =============================================================
allowed_seasons = ["2023-24", "2024-25", "2025-26"]
removed_season_filter = no0_players[~no0_players["season"].isin(allowed_seasons)].copy()
removed_season_filter["Reason"] = "Season not included in trial dataset"

# =============================================================
# COMBINE ALL REASONS
# =============================================================
removed_all = pd.concat([
    removed_zero_minutes[["season", "Player UUID", "Player Name_raw", "Reason"]],
    removed_after_target[["season", "Player UUID", "Player Name_no0", "Reason"]],
    removed_season_filter[["season", "Player UUID", "Player Name", "Reason"]]
], ignore_index=True)

# Normalize Player Name column
removed_all["Player Name"] = (
    removed_all["Player Name_raw"]
    .fillna(removed_all["Player Name_no0"])
    .fillna(removed_all["Player Name"])
)

removed_all = removed_all[["season", "Player UUID", "Player Name", "Reason"]]

# =============================================================
# SAVE REPORT
# =============================================================
OUT_DIR = Path("../output/linear_regression_model_2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = OUT_DIR / "removed_players_per_season.csv"
removed_all.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print(f"✔ Report saved successfully → {OUT_PATH}")
print("\n Preview:")
display(removed_all.head(20))


Generating report: Players Removed per Season (with reason)...



✔ Report saved successfully → ..\output\linear_regression_model_2\removed_players_per_season.csv

 Preview:


,season,Player UUID,Player Name,Reason
0,2024-25,16b72858-75e4-4125-ad3b-e13f81e6d815,Aaron Anselmino,Minutes Played = 0
1,2023-24,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,Aaron Connolly,Minutes Played = 0
2,2024-25,b3ce891a-840c-4259-95d5-28dd820c0551,Aaron Hickey,Minutes Played = 0
3,2020-21,82261583-6d63-46a0-a277-6dd0c4304c3b,Aaron Mooy,Minutes Played = 0
4,2021-22,fdddd3e0-3f83-43af-8409-f28b1d3ab27e,Aaron Ramsey,Minutes Played = 0
5,2018-19,28963763-2118-4982-bb8a-1e325dd6bdcb,Abd-Al-Ali Morakinyo Olaposi Koiki,Minutes Played = 0
6,2019-20,28963763-2118-4982-bb8a-1e325dd6bdcb,Abd-Al-Ali Morakinyo Olaposi Koiki,Minutes Played = 0
7,2023-24,1bcea839-56b3-4806-8a07-1989675c6229,Abdul Rahman Baba,Minutes Played = 0
8,2021-22,d8e1dbb9-2e08-459a-acd0-ae7a4d66f8c2,Abu Kamara,Minutes Played = 0
9,2018-19,ae9cd7e9-ae4a-46ca-aaed-03fc3ddd4579,Adalberto Peñaranda,Minutes Played = 0


In [18]:
# ===================================================================
# REPORT — Player Stats Per Season (with Web Name + Minutes + Games)
# ===================================================================

print("Generating clean player-season report...\n")

# Find Minutes Played column
minute_candidates = ["Minutes Played", "minutes", "MinutesPlayed", "mins"]
minutes_col = None

for c in minute_candidates:
    if c in df.columns:
        minutes_col = c
        break

if minutes_col is None:
    raise ValueError("❌ No Minutes Played column found in df!")

# ---------------- TRAIN & VALIDATION SUBSETS ----------------
train_df = df[df["Split"] == "Train"].copy()
val_df   = df[df["Split"] == "Validation"].copy()

# ---------------- BASE AGGREGATIONS ----------------
# Total games, total minutes, average minutes (train+val)
base_stats = (
    df.groupby(["Player UUID", "Player Name", "Web Name", "season"])
    .agg(
        Total_Games=("Gameweek", "count"),
        Total_Minutes=(minutes_col, "sum"),
        Avg_Minutes=(minutes_col, "mean")
    )
)

# Train stats per player-season
train_stats = (
    train_df.groupby(["Player UUID", "Player Name", "Web Name", "season"])
    .agg(
        Total_Train_Games=("Gameweek", "count")
    )
)

# Validation stats per player-season
val_stats = (
    val_df.groupby(["Player UUID", "Player Name", "Web Name", "season"])
    .agg(
        Total_Val_Games=("Gameweek", "count")
    )
)

# ---------------- MERGE EVERYTHING ----------------
report = base_stats.join(train_stats, how="left").join(val_stats, how="left")

# Fill missing values (players who appear only in train or only in validation)
report = report.fillna({
    "Total_Train_Games": 0,
    "Total_Val_Games": 0
})

# ---------------- CLEAN COLUMN NAMES ----------------
report = report.reset_index()

report.rename(columns={
    "Avg_Minutes": "Avg Minutes Played",
    "Total_Minutes": "Total Minutes Played",
    "Total_Games": "Total Games Played",
    "Total_Train_Games": "Total Train Games",
    "Total_Val_Games": "Total Validation Games"
}, inplace=True)

# ---------------- SAVE ----------------
OUT_DIR = Path("../output/training_reports")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = OUT_DIR / "player_stats_per_season.csv"
report.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print(f"✔ Report saved → {OUT_PATH}")
display(report.head(20))


Generating clean player-season report...

✔ Report saved → ..\output\training_reports\player_stats_per_season.csv


,Player UUID,Player Name,Web Name,season,Total Games Played,Total Minutes Played,Avg Minutes Played,Total Train Games,Total Validation Games
0,00d29d0a-98c1-4730-9e17-1d278cb660ae,Sofyan Amrabat,Amrabat,2023-24,19,771,40.578947,14.0,5.0
1,00e5b1e6-a33d-4079-aa0e-b31b4c7d2b05,Tyrique George,George,2024-25,7,97,13.857143,5.0,2.0
2,00e5b1e6-a33d-4079-aa0e-b31b4c7d2b05,Tyrique George,George,2025-26,3,137,45.666667,0.0,3.0
3,00e8f0fc-d5d0-4d9d-b9e9-507b357185d2,Leander Dendoncker,Dendoncker,2023-24,7,117,16.714286,2.0,5.0
4,017c5167-5c42-43d0-96d3-024893e8470a,Reece Burke,Burke,2023-24,19,1414,74.421053,14.0,5.0
5,01a5b46a-3914-40f3-9528-4a0eb0cf12fb,Mario Lemina,Mario Jr.,2023-24,33,2789,84.515152,33.0,0.0
6,01a5b46a-3914-40f3-9528-4a0eb0cf12fb,Mario Lemina,Mario Jr.,2024-25,16,1332,83.250000,11.0,5.0
7,01bb558f-8395-415b-aa97-1e1cdc430c8e,Emmanuel Agbadou,Agbadou,2024-25,15,1320,88.000000,15.0,0.0
8,01bb558f-8395-415b-aa97-1e1cdc430c8e,Emmanuel Agbadou,Agbadou,2025-26,7,525,75.000000,2.0,5.0
9,01d9e2c2-79d9-4ae9-9ade-008db3292ee2,Ben Osborn,Osborn,2023-24,22,1265,57.500000,17.0,5.0


In [22]:
# ===================================================================
# REPORT — Players Used in Training Per Season (Avg Minutes Played)
# ===================================================================

print("Generating TRAINING-ONLY player-season report...\n")

minute_candidates = ["Minutes Played", "minutes", "MinutesPlayed", "mins"]
minutes_col = None

for c in minute_candidates:
    if c in df.columns:
        minutes_col = c
        break

if minutes_col is None:
    raise ValueError("❌ No Minutes Played column found in df!")


# ---------------- TRAIN ONLY ----------------
train_df = df[df["Split"] == "Train"].copy()

train_stats = (
    train_df.groupby(["Player UUID", "Player Name", "Web Name", "season"])
    .agg(
        Total_Train_Games=("Gameweek", "count"),
        Total_Train_Minutes=(minutes_col, "sum"),
        Avg_Train_Minutes=(minutes_col, "mean")
    )
    .reset_index()
)

train_stats.rename(columns={
    "Total_Train_Games": "Total Games Played (Train)",
    "Avg_Train_Minutes": "Avg Minutes Played (Train)",
    "Total_Train_Minutes": "Total Minutes Played (Train)",
}, inplace=True)

# ---------------- SAVE ----------------
OUT_DIR = Path("../output/training_reports")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = OUT_DIR / "players_in_train_per_season.csv"
train_stats.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print(f"✔ Report saved → {OUT_PATH}")
display(train_stats.head(20))


Generating TRAINING-ONLY player-season report...

✔ Report saved → ..\output\training_reports\players_in_train_per_season.csv


,Player UUID,Player Name,Web Name,season,Total Games Played (Train),Total Minutes Played (Train),Avg Minutes Played (Train)
0,00d29d0a-98c1-4730-9e17-1d278cb660ae,Sofyan Amrabat,Amrabat,2023-24,14,634,45.285714
1,00e5b1e6-a33d-4079-aa0e-b31b4c7d2b05,Tyrique George,George,2024-25,5,84,16.800000
2,00e8f0fc-d5d0-4d9d-b9e9-507b357185d2,Leander Dendoncker,Dendoncker,2023-24,2,2,1.000000
3,017c5167-5c42-43d0-96d3-024893e8470a,Reece Burke,Burke,2023-24,14,1009,72.071429
4,01a5b46a-3914-40f3-9528-4a0eb0cf12fb,Mario Lemina,Mario Jr.,2023-24,33,2789,84.515152
5,01a5b46a-3914-40f3-9528-4a0eb0cf12fb,Mario Lemina,Mario Jr.,2024-25,11,882,80.181818
6,01bb558f-8395-415b-aa97-1e1cdc430c8e,Emmanuel Agbadou,Agbadou,2024-25,15,1320,88.000000
7,01bb558f-8395-415b-aa97-1e1cdc430c8e,Emmanuel Agbadou,Agbadou,2025-26,2,180,90.000000
8,01d9e2c2-79d9-4ae9-9ade-008db3292ee2,Ben Osborn,Osborn,2023-24,17,815,47.941176
9,0201278a-734f-4da2-97af-690a6575faab,Luke Berry,Berry,2023-24,10,128,12.800000
